# Ejercicio 5 — Incorporación incremental de 2024

Trabajo individual. Se verifica la descarga de doce meses de taxis amarillos y verdes de 2024, la conservación de los archivos originales de 2026 y las consultas sobre ambos años.

Antes de ejecutar este notebook, siga los comandos del README: guardar el snapshot de 2026 **antes** de ampliar, descargar los dos años, repetir la descarga y validar los metadatos con `--require-full-years 2024`. El notebook no descarga datos ni reemplaza un snapshot anterior.

Las fuentes son los Parquet locales de 2024 y 2026. Las consultas nuevas registran año de archivo, tipo y procedencia; se conservan las anomalías originales de las fechas de viaje.

In [1]:
from pathlib import Path
import os
import sys
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts" / "validate_incremental.py").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "scripts"))
from validate_incremental import run_validation, SQL_DIR
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 45)
print("Proyecto:", ROOT)
print("Vistas compartidas:\n")
print((ROOT / "sql" / "exploracion" / "00_fuentes.sql").read_text())

Proyecto: /workspace
Vistas compartidas:

-- Vistas en memoria: los viajes permanecen en los archivos Parquet.
-- union_by_name admite columnas añadidas entre meses.
-- La rama vacía declara campos opcionales incluso al leer solo 2024.
-- WHERE FALSE garantiza que no añade registros ni imputa valores.
CREATE OR REPLACE VIEW yellow_raw AS
SELECT * FROM read_parquet('data/raw/yellow/*/*.parquet', union_by_name = true, filename = true)
UNION ALL BY NAME
SELECT NULL::DOUBLE AS cbd_congestion_fee, NULL::VARCHAR AS request_source
WHERE FALSE;
CREATE OR REPLACE VIEW green_raw AS
SELECT * FROM read_parquet('data/raw/green/*/*.parquet', union_by_name = true, filename = true)
UNION ALL BY NAME
SELECT NULL::DOUBLE AS cbd_congestion_fee, NULL::VARCHAR AS request_source
WHERE FALSE;

-- Solo se armonizan nombres; no se eliminan ni imputan registros.
CREATE OR REPLACE VIEW viajes AS
SELECT 'yellow' AS tipo_taxi, filename AS archivo,
       CAST(try_strptime(regexp_extract(filename, '_([0-9]{4}-[0-9]

## 5.1–5.5: descarga incremental y preservación

`--years 2024 2026` amplía el alcance sin cambiar las rutas anteriores. Los archivos existentes no se descargan otra vez. El validador exige doce meses por tipo en 2024, mientras que 2026 conserva únicamente los meses publicados.

La siguiente celda compara SHA-256, tamaño y fecha de modificación de los archivos de 2026 con el snapshot previo. También contrasta los conteos SQL con los metadatos y ejecuta las 21 consultas anteriores. Los resultados ampliados se guardan por separado, sin sobrescribir el análisis original de 2026.

In [2]:
frames = run_validation()
manifest = json.loads((ROOT / "data" / "processed" / "download_manifest_2024_2026.json").read_text())
validation = json.loads((ROOT / "docs" / "validacion_descarga_2024_2026.json").read_text())
preservation = json.loads((ROOT / "docs" / "preservacion_2026_ejercicio_5.json").read_text())
print("Segunda descarga:", manifest["resumen"])
print("Parquet legibles:", validation["archivos_validados"])
print("Registros según metadatos:", validation["total_registros_metadata"])
print("2026 preservado:", preservation["todos_sin_cambios"], "en", preservation["archivos_preservados"], "archivos")
display(pd.DataFrame(preservation["archivos"])[["ruta", "bytes", "sin_cambios"]])

Validación: 01_consulta_conjunta


Validación: 02_inventario


Validación: 03_esquemas_por_anio


Validación: 04_campos_opcionales


Validación: 05_meses_comunes


...

Compatibilidad: sql/eda/10_importes_negativos.sql


Compatibilidad: sql/eda/11_sensibilidad.sql


Validación incremental completada.


Segunda descarga: {'descargados': 0, 'omitidos': 40, 'no_publicados': ['yellow 2026-09', 'yellow 2026-10', 'yellow 2026-11', 'yellow 2026-12', 'green 2026-09', 'green 2026-10', 'green 2026-11', 'green 2026-12'], 'fallidos': []}
Parquet legibles: 40
Registros según metadatos: 71870407
2026 preservado: True en 16 archivos


,ruta,bytes,sin_cambios
0,data/raw/green/2026/green_tripdata_2026-01.par...,991656,True
1,data/raw/green/2026/green_tripdata_2026-02.par...,920753,True
2,data/raw/green/2026/green_tripdata_2026-03.par...,1082530,True
3,data/raw/green/2026/green_tripdata_2026-04.par...,1075896,True
4,data/raw/green/2026/green_tripdata_2026-05.par...,1102947,True
5,data/raw/green/2026/green_tripdata_2026-06.par...,1075836,True
6,data/raw/green/2026/green_tripdata_2026-07.par...,1018250,True
7,data/raw/green/2026/green_tripdata_2026-08.par...,1007530,True
8,data/raw/yellow/2026/yellow_tripdata_2026-01.p...,64165080,True
9,data/raw/yellow/2026/yellow_tripdata_2026-02.p...,58683353,True


## 01_consulta_conjunta: Consulta conjunta por año y tipo

**Objetivo:** Demostrar que una misma vista consulta 2024 y 2026 y contar sus fuentes y registros.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/01_consulta_conjunta.sql`.

In [3]:
name = '01_consulta_conjunta'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])

-- Año de partición, no año de recogida: conserva las anomalías originales.
SELECT year(mes_archivo) AS anio, tipo_taxi,
       count(DISTINCT archivo) AS archivos, count(*) AS registros,
       min(mes_archivo) AS primer_mes, max(mes_archivo) AS ultimo_mes
FROM viajes WHERE year(mes_archivo) IN (2024, 2026)
GROUP BY anio, tipo_taxi ORDER BY anio, tipo_taxi;



,anio,tipo_taxi,archivos,registros,primer_mes,ultimo_mes
0,2024,green,12,660218,2024-01-01,2024-12-01
1,2024,yellow,12,41169720,2024-01-01,2024-12-01
2,2026,green,8,337114,2026-01-01,2026-08-01
3,2026,yellow,8,29703355,2026-01-01,2026-08-01


**Resultado completo:** `docs/resultados_ejercicio_5/01_consulta_conjunta.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 02_inventario: Inventario ampliado

**Objetivo:** Enumerar cada Parquet y contrastar COUNT de DuckDB con sus metadatos PyArrow.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/02_inventario.sql`.

In [4]:
name = '02_inventario'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])

-- Lista exacta de fuentes y conteo por archivo para contrastar con PyArrow.
SELECT year(mes_archivo) AS anio, tipo_taxi, mes_archivo, archivo,
       count(*) AS registros
FROM viajes WHERE year(mes_archivo) IN (2024, 2026)
GROUP BY anio, tipo_taxi, mes_archivo, archivo
ORDER BY anio, tipo_taxi, mes_archivo;



,anio,tipo_taxi,mes_archivo,archivo,registros
0,2024,green,2024-01-01,data/raw/green/2024/green_tripdata_2024-01.par...,56551
1,2024,green,2024-02-01,data/raw/green/2024/green_tripdata_2024-02.par...,53577
2,2024,green,2024-03-01,data/raw/green/2024/green_tripdata_2024-03.par...,57457
3,2024,green,2024-04-01,data/raw/green/2024/green_tripdata_2024-04.par...,56471
4,2024,green,2024-05-01,data/raw/green/2024/green_tripdata_2024-05.par...,61003
5,2024,green,2024-06-01,data/raw/green/2024/green_tripdata_2024-06.par...,54748
6,2024,green,2024-07-01,data/raw/green/2024/green_tripdata_2024-07.par...,51837
7,2024,green,2024-08-01,data/raw/green/2024/green_tripdata_2024-08.par...,51771
8,2024,green,2024-09-01,data/raw/green/2024/green_tripdata_2024-09.par...,54440
9,2024,green,2024-10-01,data/raw/green/2024/green_tripdata_2024-10.par...,56147


**Resultado completo:** `docs/resultados_ejercicio_5/02_inventario.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 03_esquemas_por_anio: Esquemas por año

**Objetivo:** Identificar columnas ausentes y cambios de tipos físicos antes de interpretar NULL.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/03_esquemas_por_anio.sql`.

In [5]:
name = '03_esquemas_por_anio'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])
print("Campos opcionales y cambios de tipos:")
display(frames[name].loc[frames[name].columna.str.lower().isin(["cbd_congestion_fee", "request_source", "ratecodeid", "payment_type", "passenger_count", "trip_type"])])

-- Presencia física de las columnas y tipos por año; NULL lógico no implica dato faltante.
SELECT CAST(regexp_extract(file_name, '_([0-9]{4})-[0-9]{2}\.parquet$', 1) AS INTEGER) AS anio,
       regexp_extract(file_name, '(yellow|green)_tripdata_', 1) AS tipo_taxi,
       name AS columna, duckdb_type AS tipo_original,
       count(DISTINCT file_name) AS archivos_con_columna
FROM parquet_schema(['data/raw/yellow/*/*.parquet', 'data/raw/green/*/*.parquet'])
WHERE num_children IS NULL
  AND regexp_extract(file_name, '_([0-9]{4})-[0-9]{2}\.parquet$', 1) IN ('2024', '2026')
GROUP BY anio, tipo_taxi, columna, tipo_original
ORDER BY tipo_taxi, columna, anio, tipo_original;



,anio,tipo_taxi,columna,tipo_original,archivos_con_columna
0,2024,green,DOLocationID,INTEGER,12
1,2026,green,DOLocationID,INTEGER,8
2,2024,green,PULocationID,INTEGER,12
3,2026,green,PULocationID,INTEGER,8
4,2024,green,RatecodeID,BIGINT,12
...,...,...,...,...,...
77,2026,yellow,tpep_dropoff_datetime,TIMESTAMP,8
78,2024,yellow,tpep_pickup_datetime,TIMESTAMP,12
79,2026,yellow,tpep_pickup_datetime,TIMESTAMP,8
80,2024,yellow,trip_distance,DOUBLE,12


Campos opcionales y cambios de tipos:


,anio,tipo_taxi,columna,tipo_original,archivos_con_columna
4,2024,green,RatecodeID,BIGINT,12
5,2026,green,RatecodeID,BIGINT,8
8,2026,green,cbd_congestion_fee,DOUBLE,8
25,2024,green,passenger_count,BIGINT,12
26,2026,green,passenger_count,BIGINT,8
27,2024,green,payment_type,BIGINT,12
28,2026,green,payment_type,BIGINT,8
29,2026,green,request_source,VARCHAR,3
40,2024,green,trip_type,BIGINT,12
41,2026,green,trip_type,BIGINT,8


**Resultado completo:** `docs/resultados_ejercicio_5/03_esquemas_por_anio.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 04_campos_opcionales: Campos opcionales y nulos

**Objetivo:** Verificar la representación de campos añadidos sin imputar datos de años anteriores.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/04_campos_opcionales.sql`.

In [6]:
name = '04_campos_opcionales'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])

-- Los campos ausentes en 2024 deben conservarse como NULL, no imputarse con cero.
SELECT year(mes_archivo) AS anio, tipo_taxi, count(*) AS registros,
       count_if(cbd_congestion_fee IS NULL) AS cbd_congestion_fee_nulo,
       count_if(request_source IS NULL) AS request_source_nulo,
       count_if(passenger_count IS NULL) AS pasajeros_nulos,
       count_if(payment_type IS NULL) AS pago_nulo
FROM viajes WHERE year(mes_archivo) IN (2024, 2026)
GROUP BY anio, tipo_taxi ORDER BY anio, tipo_taxi;



,anio,tipo_taxi,registros,cbd_congestion_fee_nulo,request_source_nulo,pasajeros_nulos,pago_nulo
0,2024,green,660218,660218.0,660218.0,24328.0,24328.0
1,2024,yellow,41169720,41169720.0,41169720.0,4091232.0,0.0
2,2026,green,337114,0.0,317903.0,48775.0,48775.0
3,2026,yellow,29703355,0.0,26799909.0,7716688.0,0.0


**Resultado completo:** `docs/resultados_ejercicio_5/04_campos_opcionales.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 05_meses_comunes: Meses con cobertura común

**Objetivo:** Identificar meses presentes para ambos taxis y ambos años, evitando comparar un año completo con uno parcial.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/05_meses_comunes.sql`.

In [7]:
name = '05_meses_comunes'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])

-- Define cobertura comparable. No compara un año completo con uno parcial.
WITH cobertura AS (
    SELECT year(mes_archivo) AS anio, month(mes_archivo) AS mes,
           count(DISTINCT tipo_taxi) AS tipos
    FROM viajes WHERE year(mes_archivo) IN (2024, 2026)
    GROUP BY anio, mes
), comunes AS (
    SELECT mes FROM cobertura WHERE tipos = 2 GROUP BY mes HAVING count(DISTINCT anio) = 2
)
SELECT c.anio, c.mes, c.tipos
FROM cobertura c JOIN comunes USING (mes)
ORDER BY c.anio, c.mes;



,anio,mes,tipos
0,2024,1,2
1,2024,2,2
2,2024,3,2
3,2024,4,2
4,2024,5,2
5,2024,6,2
6,2024,7,2
7,2024,8,2
8,2026,1,2
9,2026,2,2


**Resultado completo:** `docs/resultados_ejercicio_5/05_meses_comunes.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 06_tipos_unificados: Tipos de la vista conjunta

**Objetivo:** Documentar los tipos resultantes de la combinación de esquemas.

**Fuente:** los archivos listados en `02_inventario.csv`, mediante las vistas externas. La consulta 03 lee metadatos y la 06 describe la vista.

**SQL:** `sql/incremental/06_tipos_unificados.sql`.

In [8]:
name = '06_tipos_unificados'
print((SQL_DIR / (name + ".sql")).read_text())
display(frames[name])

-- Documenta los tipos finales tras union_by_name y UNION ALL.
DESCRIBE SELECT * FROM viajes;



,column_name,column_type,null,key,default,extra
0,tipo_taxi,VARCHAR,YES,None,None,None
1,archivo,VARCHAR,YES,None,None,None
2,mes_archivo,DATE,YES,None,None,None
3,vendor_id,INTEGER,YES,None,None,None
4,recogida,TIMESTAMP,YES,None,None,None
5,llegada,TIMESTAMP,YES,None,None,None
6,passenger_count,BIGINT,YES,None,None,None
7,trip_distance,DOUBLE,YES,None,None,None
8,ratecode_id,BIGINT,YES,None,None,None
9,store_and_fwd_flag,VARCHAR,YES,None,None,None


**Resultado completo:** `docs/resultados_ejercicio_5/06_tipos_unificados.csv`.

**Decisión:** conservar origen, diferencias de esquema y conteos; declarar año y cobertura antes de comparar. Los campos físicamente ausentes se representan con NULL, no se imputan con cero.

## 5.7: continuidad de las consultas anteriores

`union_by_name` resuelve columnas añadidas entre años. Se incorporó una rama vacía tipada para que `cbd_congestion_fee` y `request_source` existan lógicamente al consultar solo 2024, sin añadir registros. Las promociones de tipos preservan valores sin forzar enteros.

Las consultas analíticas de los ejercicios 3 y 4 siguen ejecutándose sin modificar su SQL. Esto no significa que una agregación sobre todos los archivos compare años: para ese objetivo hay que agrupar o filtrar por año. Los resultados conjuntos de compatibilidad están separados de los históricos.

In [9]:
compatibility = pd.read_csv(ROOT / "docs" / "resultados_ejercicio_5" / "compatibilidad_sql.csv")
display(compatibility)
print("Consultas compatibles:", int((compatibility.estado == "OK").sum()))

,ejercicio,consulta,estado,filas_resultado,resultado
0,3,sql/exploracion/01_inventario.sql,OK,40,docs/resultados_ejercicio_5/compatibilidad_eje...
1,3,sql/exploracion/02_esquemas.sql,OK,802,docs/resultados_ejercicio_5/compatibilidad_eje...
2,3,sql/exploracion/03_cambios_esquema.sql,OK,40,docs/resultados_ejercicio_5/compatibilidad_eje...
3,3,sql/exploracion/04_muestras.sql,OK,10,docs/resultados_ejercicio_5/compatibilidad_eje...
4,3,sql/exploracion/05_nulos.sql,OK,2,docs/resultados_ejercicio_5/compatibilidad_eje...
5,3,sql/exploracion/06_calidad.sql,OK,2,docs/resultados_ejercicio_5/compatibilidad_eje...
6,3,sql/exploracion/07_distribuciones.sql,OK,2,docs/resultados_ejercicio_5/compatibilidad_eje...
7,3,sql/exploracion/08_categorias_pago.sql,OK,12,docs/resultados_ejercicio_5/compatibilidad_eje...
8,3,sql/exploracion/09_request_source.sql,OK,61,docs/resultados_ejercicio_5/compatibilidad_eje...
9,3,sql/exploracion/10_ejemplos_alertas.sql,OK,12,docs/resultados_ejercicio_5/compatibilidad_eje...


Consultas compatibles: 21


## 5.9: diseño incremental y preparación del benchmark

Los parámetros de años, directorios por tipo/año, omisión de archivos existentes, manifiestos y vistas sobre patrones de archivos permiten ampliar el conjunto sin rehacer el flujo.

Se definen tres volúmenes para el ejercicio 6: un mes de 2024, seis meses de 2024 y todo el conjunto validado de 2024–2026. Cada nivel conserva rutas, tamaños y conteos exactos. **Todavía no se materializan tablas ni se miden tiempos.**

In [10]:
preparation = json.loads((ROOT / "docs" / "preparacion_benchmark.json").read_text())
display(pd.DataFrame([{k: level[k] for k in ("nombre", "descripcion", "cantidad_archivos", "registros", "bytes_parquet")} for level in preparation["niveles"]]))
print("Estado:", preparation["estado"])
print("Documentación: docs/ejercicio_5_incorporacion_2024.md")

,nombre,descripcion,cantidad_archivos,registros,bytes_parquet
0,pequeno,"Enero de 2024, amarillos y verdes",2,3021175,51323925
1,mediano,"Enero a junio de 2024, amarillos y verdes",12,20671900,350080949
2,completo,Todos los archivos validados de 2024 y 2026,40,71870407,1228648745


Estado: datos_preparados_sin_mediciones_de_rendimiento
Documentación: docs/ejercicio_5_incorporacion_2024.md


## Límites de comparación

2024 tiene doce meses y 2026 solo los disponibles. La consulta 05 identifica meses comunes para ambos tipos; un total anual de 2024 no es directamente comparable con el total parcial de 2026.

Los nulos estructurales de columnas nuevas no equivalen a valores cero. La cobertura de pago o pasajeros puede cambiar entre años; las diferencias requieren interpretación. Los percentiles aproximados de las consultas existentes pueden variar ligeramente al ejecutarse otra vez.

Los archivos de datos, manifiestos locales y snapshot siguen excluidos de Git; el notebook, SQL, resultados resumidos y documentación sí forman parte del trabajo reproducible.